<a href="https://colab.research.google.com/github/DeepFluxion/Mack_2026_Language_Analitycs/blob/main/notebooks/sessao_2_reshaping_and_pivot_tables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reshaping and Pivot Tables — pandas 3.0.3

O pandas fornece métodos para manipular `Series` e `DataFrame` e alterar a representação dos dados para processamento ou sumarização posterior.

| Função / Método | Descrição |
|---|---|
| `pivot()` / `pivot_table()` | Agrupa valores únicos dentro de uma ou mais categorias discretas |
| `stack()` / `unstack()` | Pivota um nível de coluna ou linha para o eixo oposto |
| `melt()` / `wide_to_long()` | Converte um DataFrame largo para o formato longo |
| `get_dummies()` / `from_dummies()` | Conversões com variáveis indicadoras |
| `explode()` | Converte uma coluna de listas em linhas individuais |
| `crosstab()` | Calcula uma tabulação cruzada de múltiplos arrays de fatores |
| `cut()` | Transforma variáveis contínuas em valores discretos e categóricos |
| `factorize()` | Codifica variáveis unidimensionais em rótulos inteiros |

In [ ]:
import datetime
import numpy as np
import pandas as pd

---
## 1. `pivot()` e `pivot_table()`

### 1.1 `pivot()`

Dados são frequentemente armazenados em formato chamado "empilhado" (*stacked*) ou "registro" (*record*).

- **Formato largo (*wide* / *record*):** uma linha por sujeito.
- **Formato longo (*long* / *stacked*):** múltiplas linhas por sujeito.

Para operações de séries temporais, é mais útil ter as variáveis únicas como colunas e as datas como índice. O método `DataFrame.pivot()` faz essa transformação:

In [ ]:
# Dados em formato longo: cada data aparece 4 vezes (uma por variável A, B, C, D)
data = {
    "value": range(12),
    "variable": ["A"] * 3 + ["B"] * 3 + ["C"] * 3 + ["D"] * 3,
    "date": pd.to_datetime(["2020-01-03", "2020-01-04", "2020-01-05"] * 4)
}

df = pd.DataFrame(data)
df

In [ ]:
# pivot(): transforma do formato longo para o largo
# index → linhas, columns → colunas, values → células
pivoted = df.pivot(index="date", columns="variable", values="value")
pivoted

Se o argumento `values` for omitido e o DataFrame tiver mais de uma coluna de valores, o resultado terá **colunas hierárquicas** (MultiIndex) cujo nível superior indica a coluna de valor correspondente:

In [ ]:
# Adiciona uma segunda coluna de valores
df["value2"] = df["value"] * 2

# Sem 'values': todas as colunas de valor viram o primeiro nível do MultiIndex
pivoted = df.pivot(index="date", columns="variable")
pivoted

In [ ]:
# Selecionando um subconjunto do DataFrame pivotado pelo nível superior
pivoted["value2"]

> **Nota:** `pivot()` só funciona com linhas únicas especificadas por `index` e `columns`. Se os dados contiverem duplicatas, use `pivot_table()`.

---
### 1.2 `pivot_table()`

Enquanto `pivot()` serve para pivotamento geral com vários tipos de dados, `pivot_table()` é voltado para **pivotamento com agregação de dados numéricos**, semelhante às tabelas dinâmicas de planilhas.

In [ ]:
# DataFrame base com múltiplas colunas categóricas e numéricas
df = pd.DataFrame(
    {
        "A": ["one", "one", "two", "three"] * 6,
        "B": ["A", "B", "C"] * 8,
        "C": ["foo", "foo", "foo", "bar", "bar", "bar"] * 4,
        "D": np.random.randn(24),
        "E": np.random.randn(24),
        "F": [datetime.datetime(2013, i, 1) for i in range(1, 13)]
             + [datetime.datetime(2013, i, 15) for i in range(1, 13)],
    }
)
df

In [ ]:
# pivot_table básica: média de 'D' agrupada por ['A','B'] nas linhas e 'C' nas colunas
pd.pivot_table(df, values="D", index=["A", "B"], columns=["C"])

In [ ]:
# Múltiplos valores, múltiplas colunas de agrupamento e função de agregação explícita
pd.pivot_table(
    df,
    values=["D", "E"],
    index=["B"],
    columns=["A", "C"],
    aggfunc="sum",
)

In [ ]:
# Múltiplas funções de agregação ao mesmo tempo: sum e mean
pd.pivot_table(
    df,
    values="E",
    index=["B", "C"],
    columns=["A"],
    aggfunc=["sum", "mean"],
)

In [ ]:
# Sem 'values': inclui todas as colunas numéricas em nível hierárquico adicional
pd.pivot_table(df[["A", "B", "C", "D", "E"]], index=["A", "B"], columns=["C"])

#### Usando `Grouper` no `pivot_table`

O `Grouper` pode ser usado nos argumentos `index` e `columns` para agrupar por frequência de tempo:

In [ ]:
# Grouper com freq='ME': agrupa por fim de mês usando a coluna 'F' como chave temporal
pd.pivot_table(
    df,
    values="D",
    index=pd.Grouper(freq="ME", key="F"),
    columns="C"
)

#### Adicionando margens (`margins=True`)

Passar `margins=True` adiciona uma linha e coluna com o rótulo `All`, contendo os agregados parciais de cada grupo:

In [ ]:
# margins=True: adiciona totais/agregados na linha e coluna 'All'
table = df.pivot_table(
    index=["A", "B"],
    columns="C",
    values=["D", "E"],
    margins=True,
    aggfunc="std"
)
table

In [ ]:
# stack() pode ser chamado no resultado do pivot_table para exibir com MultiIndex nas linhas
table.stack()

---
## 2. `stack()` e `unstack()`

Relacionados ao `pivot()`, os métodos `stack()` e `unstack()` são projetados para trabalhar com objetos `MultiIndex`.

- **`stack()`**: "pivota" um nível dos rótulos de coluna (possivelmente hierárquicos) para o eixo das linhas, retornando um `DataFrame` com um novo nível interno de rótulos de linha.
- **`unstack()`**: operação inversa de `stack()` — "pivota" um nível do índice de linha para o eixo das colunas.

In [ ]:
# Cria DataFrame com MultiIndex de dois níveis nas linhas
tuples = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

index = pd.MultiIndex.from_arrays(tuples, names=["first", "second"])
df = pd.DataFrame(np.random.randn(8, 2), index=index, columns=["A", "B"])

# Usa apenas as 4 primeiras linhas para demonstração
df2 = df[:4]
df2

In [ ]:
# stack(): move as colunas (A, B) para o índice → formato longo
# Resultado: Series com MultiIndex de 3 níveis (first, second, coluna)
stacked = df2.stack()
stacked

In [ ]:
# unstack() padrão: desempilha o último nível do índice (as colunas A, B voltam)
stacked.unstack()

In [ ]:
# unstack(1): desempilha o nível de posição 1 ('second')
stacked.unstack(1)

In [ ]:
# unstack(0): desempilha o nível de posição 0 ('first')
stacked.unstack(0)

In [ ]:
# Também é possível referenciar níveis pelo nome em vez de por posição
stacked.unstack("second")

> **Nota:** `stack()` e `unstack()` ordenam implicitamente os níveis de índice envolvidos. Por isso, chamar `stack()` seguido de `unstack()` (ou vice-versa) produz uma **cópia ordenada** do `DataFrame` original:

In [ ]:
# Demonstra que unstack().stack() equivale a sort_index()
index = pd.MultiIndex.from_product([[2, 1], ["a", "b"]])
df = pd.DataFrame(np.random.randn(4), index=index, columns=["A"])
df

In [ ]:
# unstack().stack() devolve os dados em ordem de índice
all(df.unstack().stack() == df.sort_index())

### 2.1 Múltiplos níveis em `stack()` / `unstack()`

É possível empilhar ou desempilhar mais de um nível ao mesmo tempo, passando uma lista de níveis. A lista pode conter nomes ou números de nível, mas não uma mistura dos dois:

In [ ]:
# DataFrame com MultiIndex de 3 níveis nas colunas
columns = pd.MultiIndex.from_tuples(
    [
        ("A", "cat", "long"),
        ("B", "cat", "long"),
        ("A", "dog", "short"),
        ("B", "dog", "short"),
    ],
    names=["exp", "animal", "hair_length"],
)

df = pd.DataFrame(np.random.randn(4, 4), columns=columns)
df

In [ ]:
# stack com múltiplos níveis por nome: move 'animal' e 'hair_length' para o índice
df.stack(level=["animal", "hair_length"])

In [ ]:
# Equivalente usando posições numéricas dos níveis (1 e 2)
df.stack(level=[1, 2])

### 2.2 Dados ausentes no `unstack()`

O desempilhamento pode gerar valores ausentes se os subgrupos não tiverem o mesmo conjunto de rótulos. Por padrão, os valores ausentes são preenchidos com o valor padrão para aquele tipo de dado. O argumento `fill_value` permite especificar um valor de preenchimento customizado:

In [ ]:
# DataFrame com MultiIndex nas linhas e colunas, com subconjunto irregular
columns = pd.MultiIndex.from_tuples(
    [("A", "cat"), ("B", "dog"), ("B", "cat"), ("A", "dog")],
    names=["exp", "animal"],
)
index = pd.MultiIndex.from_product(
    [("bar", "baz", "foo", "qux"), ("one", "two")], names=["first", "second"]
)
df = pd.DataFrame(np.random.randn(8, 4), index=index, columns=columns)

# Subconjunto irregular: não todas as combinações first/second estão presentes
df3 = df.iloc[[0, 1, 4, 7], [1, 2]]
df3

In [ ]:
# unstack() padrão: NaN onde não há dados para a combinação
df3.unstack()

In [ ]:
# fill_value substitui NaN pelo valor fornecido (-1e9 neste caso)
df3.unstack(fill_value=-1e9)

---
## 3. `melt()` e `wide_to_long()`

### 3.1 `melt()`

A função `melt()` (e o método `DataFrame.melt()`) é útil para remodelar um `DataFrame` do formato **largo** para o **longo**: uma ou mais colunas são variáveis identificadoras (`id_vars`), e todas as demais colunas são "desempivotadas" para o eixo das linhas, resultando em apenas duas colunas não-identificadoras: `"variable"` e `"value"`.

Os nomes dessas colunas podem ser customizados via `var_name` e `value_name`.

In [ ]:
# DataFrame largo: cada medida (height, weight) é uma coluna
cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    }
)
cheese

In [ ]:
# melt(): 'first' e 'last' são identificadores; 'height' e 'weight' são desempivotadas
cheese.melt(id_vars=["first", "last"])

In [ ]:
# var_name personaliza o nome da coluna de variáveis (padrão: 'variable')
cheese.melt(id_vars=["first", "last"], var_name="quantity")

#### `ignore_index`

Ao transformar um `DataFrame` com `melt()`, o índice original é descartado por padrão. Para preservar os valores de índice originais, use `ignore_index=False` (isso duplicará os valores de índice):

In [ ]:
# DataFrame com MultiIndex nas linhas
index = pd.MultiIndex.from_tuples([("person", "A"), ("person", "B")])
cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    },
    index=index,
)
cheese

In [ ]:
# ignore_index=True (padrão): índice original é descartado → novo índice sequencial
cheese.melt(id_vars=["first", "last"])

In [ ]:
# ignore_index=False: índice original é preservado (duplicado para cada variável)
cheese.melt(id_vars=["first", "last"], ignore_index=False)

### 3.2 `wide_to_long()`

`wide_to_long()` é similar ao `melt()`, mas oferece mais controle sobre a correspondência de colunas por padrão de nome. É útil quando as colunas seguem um padrão como `prefixo + sufixo` (ex.: `A1970`, `A1980`, `B1970`, `B1980`):

In [ ]:
# DataFrame com colunas no padrão: prefixo (A, B) + ano (1970, 1980)
dft = pd.DataFrame(
    {
        "A1970": {0: "a", 1: "b", 2: "c"},
        "A1980": {0: "d", 1: "e", 2: "f"},
        "B1970": {0: 2.5, 1: 1.2, 2: 0.7},
        "B1980": {0: 3.2, 1: 1.3, 2: 0.1},
        "X": dict(zip(range(3), np.random.randn(3))),
    }
)
dft["id"] = dft.index
dft

In [ ]:
# wide_to_long(): stubnames=['A','B'] identifica os prefixos
# i='id' é o identificador de linha; j='year' é o sufixo que vira coluna
pd.wide_to_long(dft, ["A", "B"], i="id", j="year")

---
## 4. `get_dummies()` e `from_dummies()`

### 4.1 `get_dummies()`

Para converter variáveis categóricas de uma `Series` em variáveis *dummy* (ou indicadoras), `get_dummies()` cria um novo `DataFrame` com colunas para cada categoria única, com valores `True`/`False` indicando a presença de cada categoria por linha.

In [ ]:
# Uso básico em uma Series: cada valor único vira uma coluna booleana
df = pd.DataFrame({"key": list("bbacab"), "data1": range(6)})
pd.get_dummies(df["key"])

In [ ]:
# Alternativa: método .str.get_dummies() (retorna inteiros 0/1 em vez de booleanos)
df["key"].str.get_dummies()

#### Argumento `prefix`

O argumento `prefix` adiciona um prefixo aos nomes das colunas — útil para mesclar o resultado com o `DataFrame` original:

In [ ]:
# prefix='key': renomeia as colunas para 'key_a', 'key_b', 'key_c'
dummies = pd.get_dummies(df["key"], prefix="key")
dummies

In [ ]:
# Join do DataFrame original com as dummies geradas
df[["data1"]].join(dummies)

#### `get_dummies()` com `cut()`

Esta função é frequentemente utilizada junto com funções de discretização como `cut()`:

In [ ]:
# cut() discretiza valores contínuos em intervalos; get_dummies() codifica os intervalos
values = np.random.randn(10)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
pd.get_dummies(pd.cut(values, bins))

#### `get_dummies()` em um `DataFrame`

Ao receber um `DataFrame`, `get_dummies()` codifica por padrão apenas as colunas do tipo `object`, `string` ou `category`, mantendo as demais inalteradas. O argumento `columns` permite especificar explicitamente quais colunas codificar:

In [ ]:
# Colunas A e B são str → codificadas; coluna C é int → mantida
df = pd.DataFrame({"A": ["a", "b", "a"], "B": ["c", "c", "b"], "C": [1, 2, 3]})
pd.get_dummies(df)

In [ ]:
# columns=['A']: codifica apenas a coluna A, independente do tipo
pd.get_dummies(df, columns=["A"])

#### Opções de `prefix`: string, lista ou dicionário

O argumento `prefix` (e `prefix_sep`) pode ser passado de três formas:

In [ ]:
# String: mesmo prefixo para todas as colunas codificadas
pd.get_dummies(df, prefix="new_prefix")

In [ ]:
# Lista: prefixo diferente para cada coluna (mesma ordem das colunas codificadas)
pd.get_dummies(df, prefix=["from_A", "from_B"])

In [ ]:
# Dicionário: mapeamento explícito nome_da_coluna → prefixo
pd.get_dummies(df, prefix={"B": "from_B", "A": "from_A"})

#### Argumento `drop_first`

Para evitar **colinearidade** ao alimentar modelos estatísticos, use `drop_first=True`. Isso remove a primeira categoria, pois ela pode ser inferida pelas demais. Colunas com apenas um nível são omitidas automaticamente:

In [ ]:
# Sem drop_first: todas as k categorias
s = pd.Series(list("abcaa"))
pd.get_dummies(s)

In [ ]:
# Com drop_first=True: apenas k-1 categorias (remove 'a')
pd.get_dummies(s, drop_first=True)

In [ ]:
# Em DataFrame: coluna A tem um único valor → omitida; coluna B mantém k-1 categorias
df = pd.DataFrame({"A": list("aaaaa"), "B": list("ababc")})
pd.get_dummies(df, drop_first=True)

#### Argumento `dtype`

O argumento `dtype` permite forçar o tipo de dado das colunas dummy geradas:

In [ ]:
# dtype=np.float32: colunas dummy com tipo float32 em vez do padrão booleano
df = pd.DataFrame({"A": list("abc"), "B": [1.1, 2.2, 3.3]})
pd.get_dummies(df, dtype=np.float32).dtypes

### 4.2 `from_dummies()`

`from_dummies()` converte a saída de `get_dummies()` de volta para uma `Series` de valores categóricos a partir de valores indicadores. A codificação dummy requer apenas `k - 1` categorias; a última é a **categoria padrão** e pode ser modificada com `default_category`:

In [ ]:
# DataFrame com duas colunas dummy (prefixo 'prefix', separador '_')
df = pd.DataFrame({"prefix_a": [0, 1, 0], "prefix_b": [1, 0, 1]})
df

In [ ]:
# from_dummies(): reconstrói a Series categórica a partir das colunas dummy
# sep='_' indica o separador entre prefixo e categoria
pd.from_dummies(df, sep="_")

In [ ]:
# Com k-1 colunas: a categoria padrão (ausência de todas) pode ser definida
df_k_menos_1 = pd.DataFrame({"prefix_a": [0, 1, 0]})

# default_category='b': linha com prefix_a=0 é classificada como 'b'
pd.from_dummies(df_k_menos_1, sep="_", default_category="b")

---
## 5. `explode()`

Para uma coluna de `DataFrame` com valores do tipo lista, `explode()` transforma cada valor de lista em uma linha separada. O índice resultante é duplicado correspondendo ao índice da linha original:

In [ ]:
# DataFrame com coluna 'values' contendo listas
keys = ["panda1", "panda2", "panda3"]
values = [["eats", "shoots"], ["shoots", "leaves"], ["eats", "leaves"]]

df = pd.DataFrame({"keys": keys, "values": values})
df

In [ ]:
# Series.explode(): cada elemento da lista vira uma linha (índice original duplicado)
df["values"].explode()

In [ ]:
# DataFrame.explode(): explode a coluna diretamente no DataFrame
df.explode("values")

`Series.explode()` substitui listas vazias por `NaN` e preserva entradas escalares:

In [ ]:
# Lista vazia → NaN; escalar 'foo' → preservado como está
s = pd.Series([[1, 2, 3], "foo", [], ["a", "b"]])
s.explode()

#### `explode()` em strings separadas por vírgula

Uma string com valores separados por vírgula pode ser dividida em uma lista com `str.split()` e então explodida para novas linhas:

In [ ]:
# str.split(',') transforma a string em lista; explode() cria uma linha por valor
df = pd.DataFrame([{"var1": "a,b,c", "var2": 1}, {"var1": "d,e,f", "var2": 2}])
df.assign(var1=df.var1.str.split(",")).explode("var1")

---
## 6. `crosstab()`

Use `crosstab()` para calcular uma **tabulação cruzada** de dois ou mais fatores. Por padrão, `crosstab()` calcula uma tabela de frequência dos fatores, a menos que um array de valores e uma função de agregação sejam fornecidos.

Qualquer `Series` passada usará o atributo `name` como nome de linha/coluna, a menos que `rownames` ou `colnames` sejam especificados:

In [ ]:
# Arrays de fatores
a = np.array(["foo", "foo", "bar", "bar", "foo", "foo"], dtype=object)
b = np.array(["one", "one", "two", "one", "two", "one"], dtype=object)
c = np.array(["dull", "dull", "shiny", "dull", "dull", "shiny"], dtype=object)

# Tabulação cruzada de 'a' versus a combinação de 'b' e 'c'
pd.crosstab(a, [b, c], rownames=["a"], colnames=["b", "c"])

Se `crosstab()` receber apenas duas `Series`, produz uma **tabela de frequência**:

In [ ]:
# Tabela de frequência simples a partir de colunas de um DataFrame
df = pd.DataFrame(
    {"A": [1, 2, 2, 2, 2], "B": [3, 3, 4, 4, 4], "C": [1, 1, np.nan, 1, 1]}
)
df

In [ ]:
# Conta quantas vezes cada combinação (A, B) aparece nos dados
pd.crosstab(df["A"], df["B"])

`crosstab()` também sumariza dados **categóricos**:

In [ ]:
# Dados Categorical com categorias explícitas (incluindo as não observadas)
foo = pd.Categorical(["a", "b"], categories=["a", "b", "c"])
bar = pd.Categorical(["d", "e"], categories=["d", "e", "f"])

# Tabulação cruzada com dados categóricos: inclui todas as categorias definidas
pd.crosstab(foo, bar)